# The Box Game

See the other notebooks for details. 

In [181]:
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS

import matplotlib.pyplot as plt


In [182]:
key = jax.random.PRNGKey(42)
true_treasure = jnp.array([320.0, 750.0])
initial_location = jnp.array([500.0, 500.0])

In [183]:

def model(visited_locations, observed_distances):
    """
    visited_locations: (N, 2) array of (x, y) coordinates visited
    observed_distances: (N,) array of noisy distance measurements
    """

    # Prior: treasure location uniform on [0, 1000] x [0, 1000]
    tx = numpyro.sample("tx", dist.Uniform(0, 1000))
    ty = numpyro.sample("ty", dist.Uniform(0, 1000))
    obs_sd = 1.0

    treasure = jnp.array([tx, ty])

    # Expected distances from treasure to each visited location
    expected_distances = jnp.sqrt(
        jnp.sum((visited_locations - treasure) ** 2, axis=1)
    )

    # Likelihood: observed distance ~ Normal(true_distance, 1)
    numpyro.sample("obs", dist.Normal(expected_distances, obs_sd), obs=observed_distances)

In [184]:
N = 20
key, k1, k2, k3 = jax.random.split(key, 4)
visited_locations = jax.random.uniform(k1, shape=(N, 2), minval=0, maxval=1000)
true_distances = jnp.sqrt(jnp.sum((visited_locations - true_treasure) ** 2, axis=1))
observed_distances = true_distances + jax.random.normal(k2, shape=true_distances.shape)

In [185]:
nuts_kernel = NUTS(model)
mcmc = MCMC(nuts_kernel, num_warmup=1000, num_samples=2000, num_chains=2)
mcmc.run(k3, visited_locations, observed_distances)
mcmc.print_summary()

samples = mcmc.get_samples()
print(f"True treasure:      {true_treasure}")
print(f"Posterior mean tx:  {samples['tx'].mean():.2f}")
print(f"Posterior mean ty:  {samples['ty'].mean():.2f}")

/tmp/ipykernel_46844/2745947755.py:2: UserWarning: There are not enough devices to run parallel chains: expected 2 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(2)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(nuts_kernel, num_warmup=1000, num_samples=2000, num_chains=2)
sample: 100%|██████████| 3000/3000 [00:00<00:00, 3153.83it/s, 3 steps of size 3.76e-01. acc. prob=0.92]


                mean       std    median      5.0%     95.0%     n_eff     r_hat
        tx    319.85      0.38    319.85    319.22    320.48   2501.80      1.00
        ty    749.45      0.28    749.45    749.01    749.92   3428.54      1.00

Number of divergences: 0
True treasure:      [320. 750.]
Posterior mean tx:  319.85
Posterior mean ty:  749.45


In [186]:
def reported_distance(location, treasure, k):
    new_key, subkey = jax.random.split(k)
    noise = jax.random.normal(subkey, shape=())
    distance = jnp.sqrt(jnp.sum((location - treasure) ** 2)) + noise
    return distance, new_key

In [187]:
def get_posterior(visited_locations, observed_distances, key, init_loc=None):
    nuts_kernel = NUTS(model)
    new_key, subkey = jax.random.split(key)

    mcmc = MCMC(nuts_kernel, num_warmup=1000, num_samples=2000, num_chains=1)

    if init_loc is not None:
        init_params = {"tx": jnp.array(init_loc[0]), "ty": jnp.array(init_loc[1])}
        mcmc.run(subkey, visited_locations, observed_distances, init_params=init_params)
    else:
        mcmc.run(subkey, visited_locations, observed_distances)

    mcmc.print_summary()
    samples = mcmc.get_samples()
    return samples, new_key

In [202]:
results = {}

for i in range(10):

    if i == 0:
        new_point = jnp.array([500.0, 500.0])
    # elif i == 1:
    #     new_point = jnp.array([100.0, 100.0])
    # elif i == 2:
    #     new_point = jnp.array([900.0, 900.0])
    # elif i == 3:
    #     new_point = jnp.array([100.0, 900.0])
    # elif i == 4:
    #     new_point = jnp.array([900.0, 100.0])
    else:
        prev_results = results[i - 1]
        new_point = jnp.array([prev_results['samples']['tx'].mean(), prev_results['samples']['ty'].mean()])

    new_distance, new_key = reported_distance(new_point, true_treasure, new_key)

    results[i] = {
        'point': new_point,
        'distance' : new_distance
    }

    visited_locations  = jnp.vstack([r['point'] for r in results.values()])
    observed_distances = jnp.stack([r['distance'] for r in results.values()])


    results[i]['samples'], new_key = get_posterior(visited_locations, observed_distances, new_key)

sample: 100%|██████████| 3000/3000 [00:06<00:00, 452.54it/s, 1 steps of size 5.90e-03. acc. prob=0.93]   



                mean       std    median      5.0%     95.0%     n_eff     r_hat
        tx    503.21    218.92    506.11    208.74    808.76    457.63      1.00
        ty    485.83    215.94    471.27    190.86    785.37    419.40      1.00

Number of divergences: 0


sample: 100%|██████████| 3000/3000 [00:03<00:00, 770.91it/s, 23 steps of size 3.11e-02. acc. prob=0.79]  



                mean       std    median      5.0%     95.0%     n_eff     r_hat
        tx    455.03    102.39    439.43    301.50    595.77    115.63      1.00
        ty    785.99     20.40    793.03    751.43    808.80    120.36      1.00

Number of divergences: 4


sample: 100%|██████████| 3000/3000 [00:02<00:00, 1009.66it/s, 3 steps of size 5.63e-01. acc. prob=0.94]



                mean       std    median      5.0%     95.0%     n_eff     r_hat
        tx    595.49      1.02    595.46    593.94    597.25    873.69      1.00
        ty    793.14      0.80    793.15    791.82    794.50    920.44      1.00

Number of divergences: 0


... (10 outputs left out)

sample: 100%|██████████| 3000/3000 [00:02<00:00, 1100.98it/s, 5 steps of size 4.32e-01. acc. prob=0.92]



                mean       std    median      5.0%     95.0%     n_eff     r_hat
        tx    319.86      0.44    319.86    319.09    320.50   1552.26      1.00
        ty    749.61      0.54    749.61    748.73    750.47    983.06      1.00

Number of divergences: 0


sample: 100%|██████████| 3000/3000 [00:02<00:00, 1059.48it/s, 3 steps of size 4.17e-01. acc. prob=0.91] 


                mean       std    median      5.0%     95.0%     n_eff     r_hat
        tx    319.87      0.45    319.89    319.14    320.60   1290.36      1.00
        ty    749.62      0.59    749.63    748.60    750.51    817.08      1.00

Number of divergences: 0
